# 1. What you'll learn

        - translate a weighted score into log-odds and probability
- read standardized coefficients as a practical scorecard
- see exactly where a straight boundary fails

        **The one question this notebook answers:** How does a linear score become a probability and an interpretable decision boundary?

# 2. Setup

The data is generated from a known logistic rule, so fitted log-odds coefficients can be checked against truth rather than admired in isolation.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

We simulate 1,000 applicants from two standardized signals. The true log-odds are −0.4 + 1.3·income_signal − 1.0·debt_signal. A Bernoulli draw turns that probability into the observed approval label.

**Small example:** Scores [−2,−1,0,1,2] become probabilities about [0.12,0.27,0.50,0.73,0.88] through 1/(1+e⁻ˢ). A one-unit score increase multiplies the odds by e≈2.72.

In [ ]:
rng=np.random.default_rng(SEED); n=1000
income=rng.normal(size=n); debt=rng.normal(size=n)
true_logit=-.4+1.3*income-1.0*debt
true_probability=1/(1+np.exp(-true_logit)); approved=rng.binomial(1,true_probability)
features=["income_signal","debt_signal"]
frame=pd.DataFrame({features[0]:income,features[1]:debt,"approved":approved,"true_probability":true_probability})
print("Shape:",frame.shape); print("Approval rate:",round(frame.approved.mean(),3)); print(frame.head(3).round(3))

# 4. Explore

Before fitting, inspect balance, class overlap, and the two-dimensional geometry. Logistic regression can combine overlapping evidence, but its 0.5 boundary must remain a straight line in this feature space.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.8))
frame.approved.value_counts().sort_index().plot.bar(ax=axes[0],color=["#8aa6a3","#176b66"]); axes[0].set_title("Class balance")
sns.kdeplot(data=frame,x="income_signal",hue="approved",fill=True,common_norm=False,ax=axes[1]); axes[1].set_title("Income signal overlaps")
sns.scatterplot(data=frame.sample(500,random_state=SEED),x="income_signal",y="debt_signal",hue="approved",alpha=.55,ax=axes[2]); axes[2].set_title("Two-feature geometry")
plt.tight_layout();plt.show()

**Takeaway:** Neither class dominates enough to make accuracy meaningless, but a baseline is still required.

**Takeaway:** Income shifts odds without perfectly determining the answer.

**Takeaway:** The label transition is broadly diagonal, making a linear log-odds boundary plausible.

# 5. Prepare

We use a stratified 60/20/20 split and keep standardization inside the pipeline. Standardized coefficients answer a useful scorecard question: what happens after a one-standard-deviation feature change?

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
dev,test=train_test_split(frame,test_size=.2,stratify=frame.approved,random_state=SEED)
train,val=train_test_split(dev,test_size=.25,stratify=dev.approved,random_state=SEED)
X_train,y_train=train[features],train.approved;X_val,y_val=val[features],val.approved;X_test,y_test=test[features],test.approved
print("Shapes:",X_train.shape,X_val.shape,X_test.shape);print("Rates:",y_train.mean().round(3),y_val.mean().round(3),y_test.mean().round(3))

# 6. Train

        The model creates one weighted score, interprets it as log-odds, and passes it through the sigmoid. Maximum likelihood chooses coefficients that make observed labels probable without requiring a hard threshold during fitting.

        Conceptually, fitting follows these steps:

        1. Standardize each feature using training statistics.
2. Form intercept + weighted feature sum for every row.
3. Convert scores to probabilities with the sigmoid.
4. Adjust coefficients to reduce log loss; freeze them before test.

In [ ]:
model=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=1000,random_state=SEED))
baseline=DummyClassifier(strategy="prior",random_state=SEED)
model.fit(X_train,y_train);baseline.fit(X_train,y_train)
coef=model.named_steps["logisticregression"].coef_[0];intercept=model.named_steps["logisticregression"].intercept_[0]
scorecard=pd.DataFrame({"standardized_coefficient":coef,"odds_multiplier":np.exp(coef)},index=features)
print("Intercept:",round(intercept,3));print(scorecard.round(3))

# 7. Evaluate

ROC AUC evaluates ranking, log loss evaluates probability quality, and accuracy evaluates one 0.5 decision. The key figure overlays the learned straight boundary on the known probability surface.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
prob=model.predict_proba(X_test)[:,1];pred=prob>=.5;base=baseline.predict_proba(X_test)[:,1]
print(f"Model accuracy={accuracy_score(y_test,pred):.3f}, AUC={roc_auc_score(y_test,prob):.3f}, log loss={log_loss(y_test,prob):.3f}")
print(f"Prior baseline AUC={roc_auc_score(y_test,base):.3f}, log loss={log_loss(y_test,base):.3f}")
xx,yy=np.meshgrid(np.linspace(-3,3,180),np.linspace(-3,3,180));grid=pd.DataFrame({features[0]:xx.ravel(),features[1]:yy.ravel()})
zz=model.predict_proba(grid)[:,1].reshape(xx.shape)
fig,axes=plt.subplots(1,2,figsize=(11,4.3))
axes[0].contourf(xx,yy,zz,levels=np.linspace(0,1,11),cmap="RdBu",alpha=.65);axes[0].contour(xx,yy,zz,levels=[.5],colors="black");axes[0].scatter(X_test.iloc[:,0],X_test.iloc[:,1],c=y_test,cmap="coolwarm",s=16,edgecolor="white",linewidth=.2);axes[0].set(title="Probability surface and 0.5 boundary",xlabel=features[0],ylabel=features[1])
axes[1].hist(prob[y_test.to_numpy()==0],bins=15,alpha=.6,label="actual 0");axes[1].hist(prob[y_test.to_numpy()==1],bins=15,alpha=.6,label="actual 1");axes[1].axvline(.5,ls="--",color="black");axes[1].set(title="Test probabilities",xlabel="predicted approval probability");axes[1].legend()
plt.tight_layout();plt.savefig(Path.cwd()/"key_figure.png",dpi=160,bbox_inches="tight");plt.show()

**Takeaway:** The fitted scorecard recovers the directions of the true rule, and the straight probability boundary ranks cases well without pretending the overlapping labels are deterministic.

# 8. Break it

We replace the diagonal world with XOR: opposite corners are positive. No single straight line can separate them, so logistic regression underfits even with perfect scaling. An RBF kernel supplies the missing curved boundary for comparison.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
rng=np.random.default_rng(SEED);X_xor=rng.uniform(-1,1,(800,2));y_xor=((X_xor[:,0]*X_xor[:,1])>0).astype(int)
Xa,Xb,ya,yb=train_test_split(X_xor,y_xor,test_size=.35,stratify=y_xor,random_state=SEED)
linear=make_pipeline(StandardScaler(),LogisticRegression(random_state=SEED)).fit(Xa,ya)
curved=make_pipeline(StandardScaler(),SVC(kernel="rbf",probability=True,random_state=SEED)).fit(Xa,ya)
auc_linear=roc_auc_score(yb,linear.predict_proba(Xb)[:,1]);auc_curved=roc_auc_score(yb,curved.predict_proba(Xb)[:,1])
print(f"Linear-logit XOR AUC={auc_linear:.3f}");print(f"RBF comparison AUC={auc_curved:.3f}");print(f"Damage from wrong boundary={auc_curved-auc_linear:.3f}")

**Use this when…** a binary outcome can be explained by additive changes in log-odds and transparent coefficients or a strong baseline matter.

        **Don't use this when…** the boundary is strongly interactive or nonlinear unless you deliberately add those features, or when causal claims are being inferred from predictive coefficients.

        ## Try this

        1. Double the true debt coefficient and verify the fitted odds multiplier moves farther below one.
2. Change the threshold from .5 to .3 and compare false positives with false negatives.
3. Add the product income×debt to the XOR model; one engineered interaction should repair much of the failure.